In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "5"

In [ ]:
import torch
free, total = torch.cuda.mem_get_info()
print(f"free={free/1e9:.1f} GB  total={total/1e9:.1f} GB")
print(torch.cuda.memory_summary())

In [ ]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

In [ ]:
import copy
import io
from pathlib import Path
import yaml

import numpy as np
import pyarrow.parquet as pq
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from dataloader import patient_id_from_relpath, patient_in_val

In [ ]:
cfg = yaml.safe_load(Path(f'{repo_root}/configs/main.yaml').read_text())


In [ ]:

def clean_transform():
    return transforms.Compose([transforms.Resize((224, 224), antialias=True), transforms.ToTensor()])


class TCGACleanTileDataset(Dataset):
    # Build a (shard_idx, row_in_shard) index over the requested patient split and
    # return clean images with barcodes. `is_train=None` keeps every patient;
    # True/False select the same train / held-out slices as dataloader.TCGATileDataset.
    # `keep_patients` (iterable of 3-part barcodes) further restricts the index to a cohort.
    def __init__(self, cfg, is_train=None, transform=None, keep_patients=None):
        data = cfg["data"]
        dataset_dir = Path(data["dataset_dir"])
        self.shards = sorted(dataset_dir.glob("shard-*.parquet"))
        if not self.shards:
            raise FileNotFoundError(f"No parquet shards (shard-*.parquet) under {dataset_dir}.")
        self.transform = transform if transform is not None else clean_transform()
        # Lazy per-worker ParquetFile handles so fork-children own their file positions.
        self._readers = [None] * len(self.shards)
        keep = set(keep_patients) if keep_patients is not None else None
        in_split_shard, in_split_row, in_split_patient = [], [], []
        for shard_idx, shard_path in enumerate(self.shards):
            paths = pq.read_table(str(shard_path), columns=["path"], memory_map=True)["path"].to_pylist()
            for row_idx, p in enumerate(paths):
                patient = patient_id_from_relpath(p)
                if is_train is not None and patient_in_val(patient, data["split_seed"], data["val_fraction"]) == is_train:
                    continue
                if keep is not None and patient not in keep:
                    continue
                in_split_shard.append(shard_idx)
                in_split_row.append(row_idx)
                in_split_patient.append(patient)
        if not in_split_shard:
            raise ValueError(f"no tiles found in {dataset_dir} for is_train={is_train}")
        self.shard_of = np.asarray(in_split_shard, dtype=np.int32)
        self.row_of = np.asarray(in_split_row, dtype=np.int32)
        self.patient_of = np.asarray(in_split_patient, dtype=object)   # lets a cohort be carved out post-hoc

    def __len__(self):
        return int(self.shard_of.shape[0])

    def __getitem__(self, idx):
        idx = int(idx)
        shard_idx = int(self.shard_of[idx])
        row_idx = int(self.row_of[idx])
        reader = self._readers[shard_idx]
        if reader is None:
            reader = pq.ParquetFile(str(self.shards[shard_idx]), memory_map=True)
            self._readers[shard_idx] = reader
        rg_size = reader.metadata.row_group(0).num_rows
        table = reader.read_row_group(row_idx // rg_size, columns=["path", "jpeg"])
        rel = table["path"][row_idx % rg_size].as_py()
        jpeg_bytes = table["jpeg"][row_idx % rg_size].as_py()
        with Image.open(io.BytesIO(jpeg_bytes)) as img:
            image = self.transform(img.convert("RGB"))
        slide_stem = rel.split("/", 1)[0]
        return {
            "image": image,
            "patient_id": "-".join(slide_stem.split("-")[:3]),
            "slide_id": slide_stem,
            "sample_idx": idx,
        }


# Carve a patient cohort out of an already-indexed dataset — the shard scan in __init__
# takes minutes, so reuse it rather than rebuilding for every cohort.
def subset_by_patients(ds, keep_patients):
    keep = np.isin(ds.patient_of, np.asarray(list(set(keep_patients)), dtype=object))
    if not keep.any():
        raise ValueError("no tiles left after the patient filter — are these 3-part TCGA barcodes?")
    sub = copy.copy(ds)
    sub._readers = [None] * len(ds.shards)   # fresh handles; don't share file positions with the parent
    sub.shard_of = ds.shard_of[keep]
    sub.row_of = ds.row_of[keep]
    sub.patient_of = ds.patient_of[keep]
    return sub


In [ ]:
ds = TCGACleanTileDataset(cfg, is_train=False)

In [ ]:
loader = DataLoader(ds, batch_size=512, num_workers=4, shuffle=True, pin_memory=True)

In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from model import DinoV2ViT, PrototypeHead, newton_schulz


def load_nanopath_model(checkpoint_path, device="cuda", weights="ema"):
    """weights: 'ema' -> ckpt['model_ema'] (teacher, usual for eval)
                'model' -> ckpt['model'] (student). Returns (model, cfg)."""
    state_key = {"ema": "model_ema", "model": "model"}[weights]
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)  # payload has the config dict
    cfg = ckpt["config"]
    model = DinoV2ViT(variant=cfg["model"]["type"])
    model.load_state_dict(ckpt[state_key], strict=True)
    model.to(device).eval()
    for p in model.parameters():
        p.requires_grad = False
    return model, cfg

def load_prototype_head(checkpoint_path, device="cuda", weights="ema"):
    state_key = {"ema": "dino_head_ema", "model": "dino_head"}[weights]
    ckpt = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
    if state_key not in ckpt:
        raise KeyError(f"{state_key!r} not in checkpoint — heads are only in full checkpoints (latest.pt).")
    h = ckpt["config"]["prototype_head"]
    head_state = ckpt[state_key]
    in_dim = head_state.get("mlp.0.weight", head_state.get("mlp.weight")).shape[1]  # backbone embed_dim
    head = PrototypeHead(in_dim, h["n_prototypes"], h["hidden_dim"], h["prototype_dim"],
                         h["n_layers"], h["ns_steps"], h["orthogonal"])
    head.load_state_dict(head_state, strict=True)
    return head.to(device).eval()

@torch.no_grad()
def prototype_similarity(head, embedding, kind="cosine"):
    """embedding: model.probe_features(...) output, [in_dim] or [B, in_dim].
       kind: 'cosine' [-1,1] (default) | 'logit' (temp-scaled) | 'prob' (softmax)."""
    head.eval()
    p = next(head.parameters())
    x = embedding.to(p.device, dtype=p.dtype)
    squeeze = x.dim() == 1
    if squeeze:
        x = x.unsqueeze(0)
    x = F.normalize(head.mlp(x), dim=-1, p=2)                                   # into prototype space
    prototypes = newton_schulz(head.prototypes, steps=head.ns_steps) if head.orthogonal \
                 else F.normalize(head.prototypes, dim=-1, p=2)                 # the bank the head scores against
    cos = x @ prototypes.T
    scale = head.logit_scale.clamp(max=math.log(100)).exp()
    out = {"cosine": cos, "logit": scale * cos, "prob": (scale * cos).softmax(dim=-1)}[kind]
    return out.squeeze(0) if squeeze else out


In [ ]:
CKPT = "/data/raymond.biju/nanopath/20260703_op1_skN_swdN_entregY_orthoY_skibotN/latest.pt"
device = "cuda"

model, cfg = load_nanopath_model(CKPT, device, weights="ema")   # backbone <- model_ema
head = load_prototype_head(CKPT, device, weights="ema")   # head     <- dino_head_ema

mean = torch.tensor(cfg["data"]["mean"], device=device).view(1, 3, 1, 1)
std  = torch.tensor(cfg["data"]["std"],  device=device).view(1, 3, 1, 1)

In [ ]:
proto_feats, patient_ids = [], []
with torch.no_grad():
    for i, batch in enumerate(loader):
        if i >= 5:
            break
        x = batch["image"].to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            emb = model.probe_features((x - mean) / std)
        sims = prototype_similarity(head, emb.float())
        proto_feats.append(sims.cpu().numpy())
        patient_ids.extend(batch["patient_id"])

proto_feats = np.concatenate(proto_feats, axis=0)
patient_ids = np.array(patient_ids)
print(proto_feats.shape, patient_ids.shape)


In [ ]:
import numpy as np
import pandas as pd

meta = pd.read_csv('cbio_all_clinical.csv')

PATIENT_COL = "patient_id"
SITE_COL    = "SITE_OF_TUMOR_TISSUE"

# Normalize the CSV key to the 3-part patient barcode so it matches our patient_ids
# (handles CSVs that store longer sample/aliquot barcodes).
key = meta[PATIENT_COL].astype(str).str.split("-").str[:3].str.join("-")
site_lookup = dict(zip(key, meta[SITE_COL]))

sites = np.array([str(site_lookup.get(pid, "Unknown")) for pid in patient_ids], dtype=object)

# SANITY CHECK — if this is mostly "Unknown", PATIENT_COL is wrong.
matched = int((sites != "Unknown").sum())
print(f"matched {matched}/{len(sites)} tiles to a site")
print(pd.Series(sites).value_counts().head(12))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

MIN_TILES = 10

def short_site(s):
    return s.rsplit("-", 1)[-1].strip() if "-" in s else s   # part after the last hyphen

# 1. site x prototype mean-activation matrix (drop Unknown + sparse sites)
df = pd.DataFrame(proto_feats)
df["site"] = sites
df = df[df["site"] != "Unknown"]
keep = df["site"].value_counts()
keep = keep[keep >= MIN_TILES].index
df = df[df["site"].isin(keep)]
assert len(keep) >= 2, f"only {len(keep)} site(s) with >={MIN_TILES} tiles — raise N_BATCHES"

M = df.groupby("site").mean()                       # sites x prototypes
M = M.loc[M.mean(axis=1).sort_values().index]       # stable site order
Mc = M.subtract(M.mean(axis=0), axis=1)             # center per prototype -> selectivity

# 2. transpose -> prototypes (rows) x sites (cols); group prototype rows by most-selective site
MT = Mc.T
# row_order = np.argsort(np.argmax(MT.values, axis=1), kind="stable")
# MT = MT.iloc[row_order]

# 3. diverging heatmap: blue (below-avg) <-> gray 0 <-> red (above-avg), symmetric
FS = 22   # base font size — turn this up/down to taste

vmax = float(np.abs(MT.values).max())
cmap = LinearSegmentedColormap.from_list("blue_gray_red", ["#2a78d6", "#f0efec", "#e34948"])
norm = TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

fig, ax = plt.subplots(figsize=(0.9 * MT.shape[1] + 4, 11), dpi=130)
fig.patch.set_facecolor("white")
im = ax.imshow(MT.values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")

ax.set_xticks(range(MT.shape[1]))
ax.set_xticklabels([short_site(s) for s in MT.columns], rotation=45, ha="right", fontsize=FS, color="#0b0b0b")
ax.set_yticks([])
ax.set_ylabel(f"prototype  (n={MT.shape[0]})", color="#52514e", fontsize=FS)
ax.set_xlabel("site of tumor tissue", color="#52514e", fontsize=FS * 1.1)
ax.set_title("Prototype activation by tumor site", color="#0b0b0b", fontsize=FS * 1.25, pad=14)
for sp in ax.spines.values(): sp.set_visible(False)

cbar = fig.colorbar(im, ax=ax, fraction=0.035, pad=0.02)
cbar.set_label("mean cosine sim − prototype avg", color="#52514e", fontsize=FS)
cbar.ax.tick_params(labelsize=FS * 0.8, colors="#52514e")
cbar.outline.set_visible(False)

plt.tight_layout()
fig.savefig("heatmap.png", dpi=200, bbox_inches="tight")   # bigger fonts persist into the file you \includegraphics
plt.show()




In [ ]:
ortho_CKPT = "/data/raymond.biju/nanopath/20260703_op1_skN_swdN_entregY_orthoY_skibotN/latest.pt"
msn_CKPT = "/data/raymond.biju/nanopath/20260703_op1_skN_swdN_entregY_orthoN_skibotN/latest.pt"
device = "cuda"

ortho_head = load_prototype_head(ortho_CKPT, device, weights="ema")
msn_head = load_prototype_head(msn_CKPT, device, weights="ema")


In [ ]:
head = msn_head

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

# --- grab prototypes (P: n_proto x d). Replace `head` with your PrototypeHead variable ---
P = head.prototypes
P = P.detach().cpu().float() if isinstance(P, torch.Tensor) else torch.as_tensor(np.asarray(P)).float()

n, d = P.shape
Pn = torch.nn.functional.normalize(P, dim=1)     # unit-norm rows -> Gram = cosine similarity
G  = (Pn @ Pn.T).numpy()

# off-diagonal stats (the actual orthogonality signal)
off = G[~np.eye(n, dtype=bool)]
print(f"P shape: {n} prototypes x {d} dims   (orthonormal basis possible: {n <= d})")
print(f"off-diagonal cosine  mean|.|={np.abs(off).mean():.3f}  max|.|={np.abs(off).max():.3f}  std={np.std((off)):.3f}")

vmax = float(np.abs(off).max())
fig, ax = plt.subplots(figsize=(7, 6), dpi=130)
im = ax.imshow(G, cmap="RdBu_r", norm=TwoSlopeNorm(0.0, -vmax, vmax), interpolation="nearest")
ax.set_title(f"Prototype Gram matrix (No Ortho Constraint) \nmean|off-diag|={np.abs(off).mean():.3f}", fontsize=13)
ax.set_xlabel("prototype"); ax.set_ylabel("prototype")
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.02)
cbar.set_label("cosine similarity")
plt.tight_layout()
fig.savefig("prototype_gram.png", dpi=200, bbox_inches="tight")
plt.show()


## Prototype activation along a histology axis: LUAD vs LUSC

The tumor-site plot mixes organ, stain batch and center effects together. Lung adenocarcinoma vs lung
squamous cell carcinoma holds the organ (and largely the acquisition pipeline) fixed, so whatever
separates the two classes has to be morphology: glandular / lepidic growth and mucin on the LUAD side,
keratinization, intercellular bridges and sheet-like nests on the LUSC side.

Cohort = patients whose `HISTOLOGIC_DIAGNOSIS` is one of those two labels, restricted to the **held-out
validation split** (`is_train=False`), so no tile here was seen during pretraining.

In [ ]:
import numpy as np
import pandas as pd

PATIENT_COL = "patient_id"
HIST_COL    = "HISTOLOGIC_DIAGNOSIS"
LUNG_CLASSES = {                       # CSV label -> short name used on the plot
    "Lung Adenocarcinoma": "LUAD",
    "Lung Squamous Cell Carcinoma": "LUSC",
}

meta = pd.read_csv("cbio_all_clinical.csv")

# Normalize the CSV key to the 3-part patient barcode so it matches the dataset's patient_ids.
key = meta[PATIENT_COL].astype(str).str.split("-").str[:3].str.join("-")
hist = meta[HIST_COL].astype(str).str.strip()

cohort = pd.DataFrame({"patient_id": key, "hist": hist})
cohort = cohort[cohort["hist"].isin(LUNG_CLASSES)].drop_duplicates(["patient_id", "hist"])

# One barcode can appear on several CSV rows (multiple samples). Distinct histologies for the
# same patient are ambiguous, so drop those patients rather than letting row order decide.
conflicted = cohort["patient_id"].duplicated(keep=False)
if conflicted.any():
    print(f"dropping {cohort.loc[conflicted, 'patient_id'].nunique()} patients with conflicting histology labels")
    cohort = cohort[~conflicted]

lung_label = dict(zip(cohort["patient_id"], cohort["hist"].map(LUNG_CLASSES)))   # barcode -> LUAD/LUSC
print(f"{len(lung_label)} lung patients in the clinical table")
print(pd.Series(lung_label).value_counts())

In [ ]:
# Val-split tiles for the lung cohort only. `ds` (is_train=False) is already indexed, so carve the
# cohort out of it; rebuild from scratch only if this notebook is being run without that cell.
if "ds" in globals():
    lung_ds = subset_by_patients(ds, lung_label.keys())
else:
    lung_ds = TCGACleanTileDataset(cfg, is_train=False, keep_patients=lung_label.keys())

tile_patient = pd.Series(lung_ds.patient_of, name="patient_id")
tile_class = tile_patient.map(lung_label).rename("class")

print(f"{len(lung_ds)} val tiles from {tile_patient.nunique()} lung patients")
print(pd.DataFrame({
    "tiles": tile_class.value_counts(),
    "patients": tile_patient.groupby(tile_class).nunique(),
}))

lung_loader = DataLoader(lung_ds, batch_size=512, num_workers=4, shuffle=True, pin_memory=True)

In [ ]:
# `head` is reassigned to msn_head further down the notebook — pin it back to the ortho checkpoint
# so this section always reports the same model as the tumor-site plot regardless of run order.
head = load_prototype_head(CKPT, device, weights="ema")

MAX_BATCHES = None      # None = full lung val split; set an int to subsample (loader shuffles, so it stays unbiased)

lung_feats, lung_pids = [], []
with torch.no_grad():
    for i, batch in enumerate(lung_loader):
        if MAX_BATCHES is not None and i >= MAX_BATCHES:
            break
        x = batch["image"].to(device, non_blocking=True)
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            emb = model.probe_features((x - mean) / std)
        lung_feats.append(prototype_similarity(head, emb.float()).cpu().numpy())
        lung_pids.extend(batch["patient_id"])

lung_feats = np.concatenate(lung_feats, axis=0)        # [n_tiles, n_prototypes] cosine sims
lung_pids = np.array(lung_pids)
lung_classes = np.array([lung_label[p] for p in lung_pids], dtype=object)
print(lung_feats.shape, pd.Series(lung_classes).value_counts().to_dict())

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# "tile"    -> mean over tiles, exactly like the tumor-site plot.
# "patient" -> mean within patient, then mean over patients: stops a handful of tile-rich slides
#              from carrying a class, which matters more here with only two columns.
AGG = "patient"

df = pd.DataFrame(lung_feats)
df["class"] = lung_classes
df["patient_id"] = lung_pids

if AGG == "patient":
    per_patient = df.groupby(["class", "patient_id"]).mean()
    M = per_patient.groupby("class").mean()
else:
    M = df.drop(columns="patient_id").groupby("class").mean()

M = M.loc[["LUAD", "LUSC"]]                    # classes x prototypes
Mc = M.subtract(M.mean(axis=0), axis=1)        # center per prototype -> selectivity (2 classes => mirrored columns)

# Rows = prototypes, sorted by the LUAD->LUSC contrast so the figure reads as one gradient.
MT = Mc.T
MT = MT.loc[(MT["LUAD"] - MT["LUSC"]).sort_values(ascending=False).index]

contrast = (M.loc["LUAD"] - M.loc["LUSC"])
print(f"|LUAD-LUSC| mean cosine gap: mean={contrast.abs().mean():.4f}  max={contrast.abs().max():.4f}")
print("most LUAD-selective prototypes:", list(contrast.sort_values(ascending=False).head(5).index))
print("most LUSC-selective prototypes:", list(contrast.sort_values().head(5).index))

FS = 22
vmax = float(np.abs(MT.values).max())
cmap = LinearSegmentedColormap.from_list("blue_gray_red", ["#2a78d6", "#f0efec", "#e34948"])
norm = TwoSlopeNorm(vcenter=0.0, vmin=-vmax, vmax=vmax)

fig, ax = plt.subplots(figsize=(6, 11), dpi=130)
fig.patch.set_facecolor("white")
im = ax.imshow(MT.values, aspect="auto", cmap=cmap, norm=norm, interpolation="nearest")

ax.set_xticks(range(MT.shape[1]))
ax.set_xticklabels(MT.columns, rotation=0, ha="center", fontsize=FS, color="#0b0b0b")
ax.set_yticks([])
ax.set_ylabel(f"prototype  (n={MT.shape[0]}, sorted by contrast)", color="#52514e", fontsize=FS)
ax.set_xlabel("histologic diagnosis", color="#52514e", fontsize=FS * 1.1)
ax.set_title("Prototype activation:\nLUAD vs LUSC", color="#0b0b0b", fontsize=FS * 1.25, pad=14)
for sp in ax.spines.values(): sp.set_visible(False)

cbar = fig.colorbar(im, ax=ax, fraction=0.06, pad=0.04)
cbar.set_label("mean cosine sim − prototype avg", color="#52514e", fontsize=FS)
cbar.ax.tick_params(labelsize=FS * 0.8, colors="#52514e")
cbar.outline.set_visible(False)

plt.tight_layout()
fig.savefig("lung_heatmap.png", dpi=200, bbox_inches="tight")
plt.show()